# Qwen2.5-VL 3B — Evaluation on Kvasir-VQA-x1

**Model:** `Qwen/Qwen2.5-VL-3B-Instruct` (3B params, general multimodal)

**Metrics:** Accuracy, F1, BLEU, ROUGE-L, ECE

## 1. Install Dependencies
**After running → Runtime → Restart runtime → skip to Cell 2.**

In [ ]:
!pip install -q datasets transformers accelerate pillow pandas tqdm
!pip install -q nltk rouge-score
!pip install -q qwen-vl-utils
print("\n" + "="*60 + "\n  RESTART RUNTIME NOW, then skip to Cell 2.\n" + "="*60)

## 2. Imports & Config
**Start here after restart.**

In [ ]:
import os, json, gc, math
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

USE_DRIVE = True
NUM_SAMPLES = 20
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
MODEL_NAME = 'Qwen2.5-VL 3B'
MODEL_KEY = 'qwen'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/AI-ML-based-approaches-for-the-medical-sector'
else:
    PROJECT_DIR = '/content/medical-vqa'

DATA_DIR = os.path.join(PROJECT_DIR, 'data')
IMAGE_DIR = os.path.join(DATA_DIR, 'images')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results', 'predictions')
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_NEW_TOKENS = 128
print(f'Device: {DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Download Dataset (if USE_DRIVE = False)

In [ ]:
if not USE_DRIVE:
    from datasets import load_dataset
    os.makedirs(IMAGE_DIR, exist_ok=True)
    ds_host = load_dataset('SimulaMet-HOST/Kvasir-VQA', split='raw')
    seen = set()
    for row in tqdm(ds_host, desc='Saving images'):
        if row['img_id'] not in seen:
            row['image'].save(os.path.join(IMAGE_DIR, f"{row['img_id']}.jpg"))
            seen.add(row['img_id'])
    for split in ['train', 'test']:
        ds = load_dataset('SimulaMet/Kvasir-VQA-x1', split=split)
        records = [{'img_id': r['img_id'], 'complexity': r['complexity'],
                    'question': r['question'], 'answer': r['answer'],
                    'question_class': r['question_class']} for r in ds]
        pd.DataFrame(records).to_csv(os.path.join(DATA_DIR, f'kvasir_vqa_x1_{split}.csv'), index=False)
else:
    print('Using data from Google Drive.')

## 4. Evaluation Metrics

In [ ]:
smoother = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def compute_word_f1(pred, gt):
    p = set(pred.strip().lower().split()); g = set(gt.strip().lower().split())
    if not p or not g: return 0.0
    c = p & g
    if not c: return 0.0
    pr = len(c)/len(p); rc = len(c)/len(g)
    return 2*pr*rc/(pr+rc)

def compute_bleu(pred, gt):
    ref = gt.strip().lower().split(); hyp = pred.strip().lower().split()
    if not ref or not hyp: return 0.0
    try: return sentence_bleu([ref], hyp, smoothing_function=smoother)
    except: return 0.0

def compute_rouge_l(pred, gt):
    if not pred.strip() or not gt.strip(): return 0.0
    return rouge.score(gt.strip().lower(), pred.strip().lower())['rougeL'].fmeasure

def compute_ece(confs, accs, n_bins=10):
    if not confs: return 0.0
    bounds = np.linspace(0, 1, n_bins+1); ece = 0.0; n = len(confs)
    for i in range(n_bins):
        mask = [(bounds[i] <= c < bounds[i+1]) for c in confs]
        nb = sum(mask)
        if nb == 0: continue
        ac = np.mean([a for a,m in zip(accs,mask) if m])
        cn = np.mean([c for c,m in zip(confs,mask) if m])
        ece += (nb/n)*abs(ac-cn)
    return ece

def select_diverse_samples(df, n, seed=42):
    samples = []
    for c in sorted(df['complexity'].unique()):
        sub = df[df['complexity']==c]
        samples.append(sub.sample(n=min(max(1,n//3),len(sub)), random_state=seed))
    return pd.concat(samples).head(n)

def evaluate_single(row, pred):
    gt = str(row['answer'])
    return {'img_id': row['img_id'], 'complexity': int(row['complexity']),
            'question_class': row['question_class'], 'question': row['question'],
            'ground_truth': gt, 'prediction': pred,
            'exact_match': pred.strip().lower()==gt.strip().lower(),
            'word_f1': round(compute_word_f1(pred,gt),3),
            'bleu': round(compute_bleu(pred,gt),3),
            'rouge_l': round(compute_rouge_l(pred,gt),3)}

def compute_summary(results):
    if not results: return {'model': MODEL_NAME, 'error': 'No results'}
    n = len(results)
    em = sum(1 for r in results if r['exact_match'])
    f1s = [r['word_f1'] for r in results]
    bleus = [r['bleu'] for r in results]
    rls = [r['rouge_l'] for r in results]
    ece = compute_ece(f1s, [1.0 if r['exact_match'] else 0.0 for r in results])
    rdf = pd.DataFrame(results)
    pc = {}
    for c in sorted(rdf['complexity'].unique()):
        cdf = rdf[rdf['complexity']==c]
        pc[f'level_{c}'] = {'exact_matches': int(cdf['exact_match'].sum()), 'total': int(len(cdf)),
            'exact_accuracy': round(cdf['exact_match'].mean()*100,1),
            'avg_word_f1': round(cdf['word_f1'].mean()*100,1),
            'avg_bleu': round(cdf['bleu'].mean()*100,1),
            'avg_rouge_l': round(cdf['rouge_l'].mean()*100,1)}
    return {'model': MODEL_NAME, 'num_samples': n,
            'exact_match_accuracy': round(em/n*100,1), 'average_word_f1': round(np.mean(f1s)*100,1),
            'average_bleu': round(np.mean(bleus)*100,1), 'average_rouge_l': round(np.mean(rls)*100,1),
            'ece': round(ece*100,2), 'exact_matches': em, 'total': n, 'per_complexity': pc}

def print_summary(s):
    print(f"\n{'='*70}\n  {s['model'].upper()} — RESULTS\n{'='*70}")
    print(f"  Accuracy: {s.get('exact_match_accuracy',0):.1f}%")
    print(f"  F1:       {s.get('average_word_f1',0):.1f}%")
    print(f"  BLEU:     {s.get('average_bleu',0):.1f}%")
    print(f"  ROUGE-L:  {s.get('average_rouge_l',0):.1f}%")
    print(f"  ECE:      {s.get('ece',0):.2f}%")
    for k,v in s.get('per_complexity',{}).items():
        print(f"    {k}: EM {v['exact_matches']}/{v['total']}, F1 {v['avg_word_f1']:.1f}%, BLEU {v['avg_bleu']:.1f}%")
    print('='*70)

print('Metrics loaded.')

## 5. Prediction Grid Visualization

In [ ]:
def plot_prediction_grid(results, max_show=6):
    show = results[:max_show]
    n = len(show)
    if n == 0: return
    cols = min(3, n); rows_n = math.ceil(n/cols)
    fig, axes = plt.subplots(rows_n, cols, figsize=(7*cols, 6*rows_n))
    if rows_n==1 and cols==1: axes = np.array([axes])
    axes = np.atleast_2d(axes)
    fig.suptitle(f'{MODEL_NAME} Predictions on Kvasir-VQA-x1', fontsize=16, fontweight='bold', y=1.01)
    for i, r in enumerate(show):
        ri, ci = divmod(i, cols); ax = axes[ri][ci]
        img_path = os.path.join(IMAGE_DIR, f"{r['img_id']}.jpg")
        if os.path.exists(img_path): ax.imshow(Image.open(img_path).convert('RGB'))
        else: ax.text(0.5,0.5,'No Image',ha='center',va='center',fontsize=14,transform=ax.transAxes)
        ax.set_xticks([]); ax.set_yticks([])
        if r['exact_match']: st='EXACT ✓'; cl='#27ae60'
        elif r['word_f1']>=0.5: st='PARTIAL ~'; cl='#f39c12'
        else: st='WRONG ✗'; cl='#e74c3c'
        txt = f"{st} | F1: {r['word_f1']:.2f} | Complexity {r['complexity']}\nQ: {r['question'][:90]}\nGT: {r['ground_truth'][:70]}\nPred: {r['prediction'][:70]}"
        ax.text(0.02, 0.98, txt, transform=ax.transAxes, fontsize=7, va='top', color=cl,
                fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.75))
    for i in range(n, rows_n*cols): axes[divmod(i,cols)[0]][divmod(i,cols)[1]].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_prediction_grid.png'), dpi=150, bbox_inches='tight')
    plt.show()
print('Grid function loaded.')

## 6. Load Data & Select Samples

In [ ]:
test_df = pd.read_csv(os.path.join(DATA_DIR, 'kvasir_vqa_x1_test.csv'))
sample_df = select_diverse_samples(test_df, NUM_SAMPLES)
print(f'Test: {len(test_df)} | Selected: {len(sample_df)}')
print(sample_df['complexity'].value_counts().sort_index().to_string())

## 7. Load Qwen2.5-VL 3B

In [ ]:
print(f'[INFO] Loading {MODEL_ID}...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()
print(f'[INFO] {MODEL_NAME} loaded successfully.')

## 8. Run Inference & Evaluate

In [ ]:
from qwen_vl_utils import process_vision_info

results = []
print(f'\n[INFO] Running {MODEL_NAME} on {len(sample_df)} samples...\n')
for idx, (_, row) in enumerate(sample_df.iterrows()):
    img_path = os.path.join(IMAGE_DIR, f"{row['img_id']}.jpg")
    if not os.path.exists(img_path):
        print(f'[SKIP] {row["img_id"]}')
        continue
    question = row['question']
    try:
        messages = [
            {'role': 'system', 'content': 'You are a medical imaging expert. Answer the question about the endoscopic image concisely.'},
            {'role': 'user', 'content': [
                {'type': 'image', 'image': f'file://{img_path}'},
                {'type': 'text', 'text': question}
            ]}
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors='pt'
        ).to(model.device)
        with torch.inference_mode():
            output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        # Trim input tokens from output
        generated_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
        prediction = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    except Exception as e:
        print(f'[ERROR] {idx}: {e}')
        prediction = ''
    result = evaluate_single(row, prediction)
    results.append(result)
    st = '✓' if result['exact_match'] else '~' if result['word_f1']>=0.5 else '✗'
    print(f"[{idx+1}/{len(sample_df)}] {st} F1:{result['word_f1']:.2f} BLEU:{result['bleu']:.2f} RL:{result['rouge_l']:.2f}")
    print(f"  Q: {result['question'][:70]}")
    print(f"  GT: {result['ground_truth'][:70]}")
    print(f"  Pred: {result['prediction'][:70]}")

print(f'\n[INFO] Inference complete. {len(results)} results.')

## 9. Results Summary

In [ ]:
summary = compute_summary(results)
print_summary(summary)
pd.DataFrame(results).to_csv(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_predictions.csv'), index=False)
with open(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Results saved to {RESULTS_DIR}')

## 10. Prediction Visualization Grid

In [ ]:
plot_prediction_grid(results)

## 11. Metrics Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
metrics = ['Accuracy', 'F1', 'BLEU', 'ROUGE-L', 'ECE']
values = [summary.get('exact_match_accuracy',0), summary.get('average_word_f1',0),
          summary.get('average_bleu',0), summary.get('average_rouge_l',0), summary.get('ece',0)]
colors = ['#3498db','#2ecc71','#9b59b6','#e67e22','#e74c3c']
bars = ax.bar(metrics, values, color=colors, edgecolor='white', width=0.6)
for b,v in zip(bars,values): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{v:.1f}%', ha='center', fontweight='bold')
ax.set_ylabel('Score (%)')
ax.set_title(f'{MODEL_NAME} — All Metrics on Kvasir-VQA-x1', fontweight='bold')
ax.set_ylim(0, max(max(values)*1.3, 10))
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{MODEL_KEY}_metrics_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12. Cleanup

In [ ]:
del model, processor
if torch.cuda.is_available(): torch.cuda.empty_cache()
gc.collect()
print('GPU memory cleared.')